# GGA 01 — Embedding Pipeline (file production)

Produces the on-disk artifacts that `gga_02_analysis.ipynb` consumes:
`narratives.csv` (filtered, masked corpus with all analysis columns),
`typology.gdb` (points, embedding feature classes), `X.npy` + `pk_order.csv`
(the 384-d embedding matrix and its row order), `progress.json`.

**If you have already run the full corpus, you do not need this notebook again**
— go straight to `gga_02_analysis.ipynb`, which reads the existing files.
Re-run this only for fresh data pulls or a changed cohort definition.

## How to run (Pro notebooks can jam — read first)
- Run cells one at a time, never "Run All".
- Heavy cells print a start timestamp + expected duration, save results to disk,
  and record themselves in `progress.json`; re-running a finished cell skips itself.
- If a cell hangs far past its expected duration: close ArcGIS Pro (Task Manager
  if needed), reopen, run cell 0 only, continue from the first unchecked step.
- Keep layers out of the map while running — open layers hold schema locks.


## 0 — Config and progress tracker
Edit the four paths. Safe to re-run any time; prints the checklist.

In [ ]:
# ========================== EDIT THESE FOUR PATHS ==========================
HOME      = r"E:\GGA\Embeddings\typology_full"                # working folder (created if missing)
CANADA    = r"E:\GGA\data\Canada_incidents.xlsx"              # BikeMaps Canada export (fresh pull)
US        = r"E:\GGA\data\United_States_incidents.xlsx"       # BikeMaps US export (fresh pull)
TEXT_DLPK = r"E:\GGA\Embeddings\TextEncoder.dlpk"             # auto-downloaded in step 2 if missing
# ===========================================================================
SAMPLE_N  = None    # None = full corpus; set e.g. 400 for a quick MVP (use a different HOME per setting!)

import os, re, json, time, random
import arcpy
import numpy as np
import pandas as pd

SEED = 1208
random.seed(SEED); np.random.seed(SEED)

os.makedirs(HOME, exist_ok=True)
GDB      = os.path.join(HOME, "typology.gdb")
PROGRESS = os.path.join(HOME, "progress.json")
STEPS    = ["1_prep", "2_points", "3_model", "4_embed", "5_extract"]

def _load():
    try:
        with open(PROGRESS) as f:
            return json.load(f)
    except Exception:
        return {}

def done(step):
    return _load().get(step, {}).get("done", False)

def mark(step, **info):
    p = _load()
    p[step] = {"done": True, "at": time.strftime("%Y-%m-%d %H:%M:%S"), **info}
    with open(PROGRESS, "w") as f:
        json.dump(p, f, indent=1)
    print(f"[done] {step}", flush=True)

def begin(step, expected):
    print(f"[start] {step} at {time.strftime('%H:%M:%S')} — expected {expected}", flush=True)
    return time.time()

def status():
    p = _load()
    print("Progress:", flush=True)
    for s in STEPS:
        print(("  [x] " if p.get(s, {}).get("done") else "  [ ] ") + s, flush=True)

if not arcpy.Exists(GDB):
    arcpy.management.CreateFileGDB(HOME, "typology.gdb")
arcpy.env.workspace = GDB
arcpy.env.overwriteOutput = True
print("ArcGIS Pro", arcpy.GetInstallInfo()["Version"], "| license:", arcpy.ProductInfo(), flush=True)
status()

## 1 — Load, filter, mask (~30 s)

Filters: date > 2016-11-30 and <= today (clamps rider typos), personally
involved, known ebike, narrative >= 20 chars, non-French (French-stopword
detector; an English-stopword test false-flags terse English). `injured` = any
injury vs none.

**Masking:** 30% of injured riders' narratives contain outcome words vs 6% of
uninjured (P(injured | outcome word) = 51% vs 17% base). Every sentence with
outcome vocabulary is deleted before embedding, so downstream clusters form
around circumstances, not outcome descriptions. Event words (hit, fell) are
deliberately kept — mechanism, not outcome.

`narratives.csv` now carries ALL columns the analysis notebook needs, so it
never has to re-open the raw Excel files.

In [ ]:
STEP = "1_prep"
PREP_CSV = os.path.join(HOME, "narratives.csv")
KEEP = ["pk", "country", "x", "y", "ebike", "injured", "details", "details_masked",
        "injury", "p_type", "i_type", "date", "age", "gender", "ebike_class",
        "incident_with", "intersection", "turning", "terrain", "road_conditions",
        "sightlines", "cars_on_roadside", "helmet", "trip_purpose"]

if done(STEP) and os.path.exists(PREP_CSV):
    narr = pd.read_csv(PREP_CSV)
    print(f"skipped — already done ({len(narr)} rows)", flush=True)
else:
    t0 = begin(STEP, "~30 s")
    frames = []
    for country, path in [("CA", CANADA), ("US", US)]:
        t = pd.read_excel(path)
        t["country"] = country
        frames.append(t)
        print(f"  read {country}: {len(t)} reports", flush=True)
    df = pd.concat(frames, ignore_index=True).drop_duplicates(subset="pk")
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    d = df[(df["date"] > "2016-11-30")
           & (df["date"] <= pd.Timestamp.now())
           & (df["personal_involvement"] == "Yes")
           & df["ebike"].isin(["Yes", "No"])].copy()
    d = d[d["details"].notna()
          & (d["details"].astype(str).str.len() >= 20)
          & d["x"].notna() & d["y"].notna()]

    FR = {"le", "la", "les", "un", "une", "des", "et", "je", "j'ai", "sur", "pour",
          "dans", "est", "était", "vélo", "voiture", "rue", "piste", "cyclable",
          "mais", "avec", "qui", "que", "pas", "très", "du", "au"}
    def is_french(s):
        words = re.findall(r"[a-zàâçéèêëîïôûùüÿœ']+", str(s).lower())
        if not words:
            return False
        hits = sum(1 for w in words if w in FR)
        return hits >= 3 and hits / len(words) > 0.08
    n_before = len(d)
    d = d[~d["details"].apply(is_french)]
    print(f"  dropped {n_before - len(d)} French narratives", flush=True)

    d["injured"] = ((~d["injury"].isin(["No injury", "Unknown"]))
                    & d["injury"].notna()).astype(int)

    LEAK = re.compile(
        r"\b(injur\w*|hospital\w*|doctor|ambulance|paramedic|brok\w*|fractur\w*"
        r"|pain\w*|bruis\w*|stitch\w*|concuss\w*|emergency|x-?ray|sprain\w*"
        r"|bleed\w*|blood|scrape\w*|road ?rash|wound\w*|surgery|clinic|e\.?r\.?)\b",
        re.I)
    def mask_text(s):
        sentences = re.split(r"(?<=[.!?])\s+", str(s))
        kept = [x for x in sentences if not LEAK.search(x)]
        return " ".join(kept).strip() or "no description"
    d["details_masked"] = d["details"].astype(str).apply(mask_text)
    d = d[d["details_masked"] != "no description"]
    print(f"  {len(d)} narratives survive masking", flush=True)

    if SAMPLE_N and len(d) > SAMPLE_N:
        total = len(d)
        parts = [g.sample(min(int(round(SAMPLE_N * len(g) / total)), len(g)), random_state=SEED)
                 for _, g in d.groupby("injured")]
        d = pd.concat(parts)
        print(f"  stratified sample: {len(d)} rows", flush=True)

    narr = d[[c for c in KEEP if c in d.columns]].reset_index(drop=True)
    narr.to_csv(PREP_CSV, index=False, encoding="utf-8-sig")
    print(f"{len(narr)} narratives | injured {narr.injured.mean():.1%} | {time.time()-t0:.0f}s", flush=True)
    mark(STEP, rows=len(narr))

## 2 — Points + text model (~10 s, plus a one-time model download)

In [ ]:
STEP = "2_points"
FC = os.path.join(GDB, "narratives")
if done(STEP) and arcpy.Exists(FC):
    print("2_points skipped — already done", flush=True)
else:
    t0 = begin(STEP, "~10 s")
    arcpy.management.XYTableToPoint(PREP_CSV, FC, "x", "y",
                                    coordinate_system=arcpy.SpatialReference(4326))
    n = int(arcpy.management.GetCount(FC)[0])
    print(f"{n} point features | {time.time()-t0:.0f}s", flush=True)
    mark(STEP, n=n)

STEP = "3_model"
if os.path.exists(TEXT_DLPK):
    print("model on disk:", TEXT_DLPK, flush=True)
    if not done(STEP):
        mark(STEP, path=TEXT_DLPK)
else:
    t0 = begin(STEP, "2-10 min (one-time download)")
    from arcgis.gis import GIS
    gis = GIS("home")
    print("  signed in as", gis.users.me.username, flush=True)
    hits = gis.content.search('title:"HF Text Feature Extraction" owner:esri_analytics',
                              item_type="Deep Learning Package") or \
           gis.content.search('title:"HF Text Feature Extraction"',
                              item_type="Deep Learning Package", outside_org=True)
    assert hits, "Living Atlas item not found — check sign-in/subscription"
    TEXT_DLPK = hits[0].download(save_path=os.path.dirname(TEXT_DLPK) or HOME)
    print("  downloaded ->", TEXT_DLPK, flush=True)
    mark(STEP, path=TEXT_DLPK)

## 3 — Embed the masked narratives (the long cell)
Full corpus on CPU: budget ~30-60 min. The tool prints nothing until it
finishes — judge by the clock. If Pro dies mid-run, re-run this cell after
restarting: it clears its own partial output first.

In [ ]:
STEP = "4_embed"
EMB_FC = os.path.join(GDB, "narratives_emb")
if done(STEP) and arcpy.Exists(EMB_FC):
    print("skipped — already done", flush=True)
else:
    if arcpy.Exists(EMB_FC):
        arcpy.management.Delete(EMB_FC)
        print("  deleted partial output from an earlier run", flush=True)
    t0 = begin(STEP, "~30-60 min full corpus on CPU")
    arcpy.geoai.GenerateEmbeddingsUsingAIModels(
        FC, EMB_FC, TEXT_DLPK,
        [["text_field", "details_masked"], ["batch_size", "16"]])
    fields = {f.name: f.type for f in arcpy.ListFields(EMB_FC)}
    assert fields.get("Embedding") == "Blob", "no Blob field named Embedding"
    print(f"embedded {arcpy.management.GetCount(EMB_FC)[0]} narratives "
          f"| {(time.time()-t0)/60:.1f} min", flush=True)
    mark(STEP)

## 4 — Unpack to a numeric matrix (~1-3 min) → X.npy + pk_order.csv

In [ ]:
STEP = "5_extract"
FLAT_FC = os.path.join(GDB, "narratives_emb_flat")
X_NPY   = os.path.join(HOME, "X.npy")
PK_CSV  = os.path.join(HOME, "pk_order.csv")
if done(STEP) and os.path.exists(X_NPY):
    print("skipped — already done", flush=True)
else:
    t0 = begin(STEP, "~1-3 min (adds ~384 fields)")
    if arcpy.Exists(FLAT_FC):
        arcpy.management.Delete(FLAT_FC)
    arcpy.geoai.ExtractEmbeddingToFields(EMB_FC, FLAT_FC, "Embedding", "emb")
    emb_cols = [f.name for f in arcpy.ListFields(FLAT_FC)
                if f.name.lower().startswith("emb") and f.type in ("Double", "Single")]
    emb_cols.sort(key=lambda c: int(re.sub(r"\D", "", c) or 0))
    rows, pks = [], []
    with arcpy.da.SearchCursor(FLAT_FC, ["pk"] + emb_cols) as cur:
        for r in cur:
            pks.append(r[0]); rows.append(r[1:])
    X = np.asarray(rows, dtype=np.float32)
    np.save(X_NPY, X)
    pd.DataFrame({"pk": np.asarray(pks)}).to_csv(PK_CSV, index=False)
    print(f"X {X.shape} saved | {time.time()-t0:.0f}s", flush=True)
    mark(STEP, dims=int(X.shape[1]))
status()
print("\nPipeline complete — open gga_02_analysis.ipynb.")